# Low-ozone composites and bootstrap analysis

Raw inputs are read from `PAPER1_ARCHIVE_ROOT`; preprocessing products are read from `PAPER1_PREPROCESSED_ROOT`; new diagnostics are written to `PAPER1_DERIVED_ROOT` (default: repository-local `work/`). Source files are never modified.


## Shared schema and bootstrap contract

Inputs: canonical rankings, staged Z300, and canonical full-latitude upward EP flux. Outputs: schema/write helpers. Method: November--March has 151 no-leap days and every random composite has the observed low-group size.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "analysis",
        Path.cwd() / "Paper1" / "analysis",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/analysis/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARCH_HINDCAST_ROOT = Path(os.environ.get(
    "PAPER1_MARCH_HINDCAST_SOURCE",
    ""
    "",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only March hindcast root:", MARCH_HINDCAST_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)

from figure15_combined import (
    bootstrap_and_package, compute_ep_calendar_standardized,
    compute_low25_composites, compute_z300_monthly_stationary,
    prepare_figure15_source,
)
from paper1_diagnostics import parse_year

def annual_z_map(root):
    output = {}
    for path in sorted(Path(root).glob("*.Z3.nc")):
        year = parse_year(path)
        if year in output:
            raise RuntimeError(f"duplicate Z3 year {year:04d} under {root}")
        output[year] = path
    return output

FIG15_REQUIRED = {
    "z300_low25_anomaly": ("month", "lat", "lon"),
    "z300_stationary_climatology": ("month", "lat", "lon"),
    "z300_bootstrap_mean": ("month", "lat", "lon"),
    "z300_bootstrap_std": ("month", "lat", "lon"),
    "z300_bootstrap_significant": ("month", "lat", "lon"),
    "epflux_low25_std_anomaly": ("pressure", "season_day"),
    "epflux_bootstrap_mean": ("pressure", "season_day"),
    "epflux_bootstrap_std": ("pressure", "season_day"),
    "epflux_bootstrap_significant": ("pressure", "season_day"),
    "is_low25": ("event_year",), "o3_minimum_du": ("event_year",),
    "canonical_rank": ("event_year",),
}

def write_figure15(dataset, name):
    dataset.attrs["product_version"] = PRODUCT_VERSION
    exact_sizes = {"month": 5, "season_day": 151, "pressure": 18}
    write_netcdf_atomic(
        dataset, product_path("figure15", name), required_vars=FIG15_REQUIRED,
        required_coords=(
            "month", "lat", "lon", "pressure", "season_day", "event_year",
            "event_id", "source_segment", "model_year", "ranking_event_year",
        ),
        exact_sizes=exact_sizes,
        required_attrs={
            "bootstrap_replicates": 5000, "ep_standardization_ddof": 0,
            "available_event_count": int(dataset.sizes["event_year"]),
            "available_low_count": int(dataset.is_low25.sum()),
            "master_sample_size": int(dataset.attrs["master_sample_size"]),
            "master_low_count": int(dataset.attrs["master_low_count"]),
            "natural_month_n2": "True", "do_ubar": "True",
            "w_argument": "None", "wave": "-1",
            "z300_completeness": str(dataset.attrs["z300_completeness"]),
            "field_scope_segments": str(dataset.attrs["field_scope_segments"]),
            "scope_master_count": int(dataset.attrs["scope_master_count"]),
            "scope_master_low_count": int(dataset.attrs["scope_master_low_count"]),
            "excluded_master_event_ids": str(dataset.attrs["excluded_master_event_ids"]),
        }, overwrite=OVERWRITE,
    )


## MERRA-2 event assembly

Inputs: independent 1980--2025 MERRA ranking, staged Z300 and EP. Outputs: untransformed event state. Method: retain only complete Nov--Mar events without changing membership.


In [ ]:
merra_rank_path = product_path("ozone", "merra2_rankings.csv")
merra_rankings = pd.read_csv(merra_rank_path)
merra_z_files = {
    year: PREPROCESSED_ROOT / "MERRA2_Processed" / "Z3" / f"MERRA2.Z3.{year}.nc"
    for year in range(1980, 2026)
}
missing = [str(path) for path in merra_z_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"MERRA-2 Z300 source is incomplete: {missing[:3]}")
merra_state = prepare_figure15_source(
    label="MERRA2", rankings=merra_rankings,
    sources={
        "MERRA2": {
            "z_files": merra_z_files,
            "z_root": PREPROCESSED_ROOT / "MERRA2_Processed" / "Z3",
            "ep_path": product_path("epflux", "merra2_1980_2025_epflux.nc"),
            "model_year": False,
        },
    },
    master_ranking_path=merra_rank_path, require_all_ranked=False,
)


## MERRA-2 Z300 monthly anomaly and stationary wave

Inputs: daily 300-hPa fields for every required November--March source day. Outputs: centered Z300 and stationary climatology. Method: first require every retained spatial grid cell to be finite on every daily source, form natural-month means with skipna=False, subtract the all-event monthly climatology, and remove zonal mean only for stationary context.


In [ ]:
compute_z300_monthly_stationary(merra_state)


## MERRA-2 EP standardization

Inputs: assembled full-latitude upward EP events. Outputs: pressure/calendar-day standardized 40--80N cosine mean. Method: standardize across events with population ddof=0 at every pressure and day; the fixed ddof is recorded because Methods V7 does not state it.


In [ ]:
compute_ep_calendar_standardized(merra_state)


## MERRA-2 fixed-low25 composites

Inputs: standardized EP and centered Z300 plus independent low25 flags. Outputs: observed low-group composites. Method: average only the fixed selected events.


In [ ]:
compute_low25_composites(merra_state)


## MERRA-2 5000-composite bootstrap

Inputs: observed composites and all-event pools. Outputs: figure15/merra2_bootstrap5000.nc. Method: 5000 same-size samples with replacement and two-standard-deviation mask.


In [ ]:
merra_output = bootstrap_and_package(merra_state, repetitions=5000, seed=15500)
write_figure15(merra_output, "merra2_bootstrap5000.nc")


## WACCM combined classification master and LONGRUN field scope

Inputs: the strict 207+23 WACCM classification master and canonical LONGRUN Z300/EP only. Outputs: a 207-event LONGRUN scope with the 23 BWCN master IDs explicitly recorded as scope exclusions and incomplete LONGRUN fields recorded separately. Method: inherit the combined-master threshold, ranks, and flags unchanged; never treat BWCN as missing and never re-rank LONGRUN.


In [ ]:
waccm_rank_path = product_path("ozone", "waccm_master_rankings.csv")
waccm_master_rankings = pd.read_csv(waccm_rank_path)
master_segment_counts = waccm_master_rankings.groupby(
    waccm_master_rankings.source_segment.astype(str)
).size().to_dict()
if len(waccm_master_rankings) != 230 or master_segment_counts != {"BWCN": 23, "LONGRUN": 207}:
    raise RuntimeError(f"Figure 15 WACCM classification master differs: {master_segment_counts}")
waccm_rankings = waccm_master_rankings.loc[
    waccm_master_rankings.source_segment.astype(str) == "LONGRUN"
].copy()
long_z_root = PREPROCESSED_ROOT / "B2000WCN001002_timefixed" / "interpolated" / "Z3"
waccm_sources = {
    "LONGRUN": {
        "z_files": annual_z_map(long_z_root), "z_root": long_z_root,
        "ep_path": product_path("epflux", "waccm_longrun_epflux.nc"),
        "model_year": True,
    },
}
if not waccm_sources["LONGRUN"]["z_files"]:
    raise FileNotFoundError("Staged LONGRUN pressure-level Z3 inputs are required")
waccm_state = prepare_figure15_source(
    label="WACCM", rankings=waccm_rankings, sources=waccm_sources,
    master_ranking_path=waccm_rank_path, require_all_ranked=False,
    classification_master=waccm_master_rankings,
)


## WACCM Z300 monthly anomaly and stationary wave

Inputs: staged daily pressure-level Z300 events. Outputs: centered Z300 and stationary climatology. Method: apply the identical all-days/all-grid finite gate and skipna=False natural-month calculation used for MERRA-2.


In [ ]:
compute_z300_monthly_stationary(waccm_state)


## WACCM EP standardization

Inputs: monthly-natural-calendar-N2 full-latitude upward EP events. Outputs: pressure/calendar-day standardized 40--80N cosine mean. Method: identical population ddof=0 standardization to MERRA-2.


In [ ]:
compute_ep_calendar_standardized(waccm_state)


## WACCM fixed-low25 composites

Inputs: standardized complete LONGRUN fields and their inherited combined-master flags. Outputs: the observed available-low LONGRUN composite. Method: average only available LONGRUN events already flagged by the 230-event master; BWCN remains outside field scope and the subset is never re-ranked.


In [ ]:
compute_low25_composites(waccm_state)


## WACCM 5000-composite bootstrap

Inputs: the observed available-low LONGRUN composite and 206-event complete-field pool. Outputs: figure15/waccm_bootstrap5000.nc. Method: each of 5000 samples draws the actual available-low count (server regression 51); metadata separately records combined master N=230/low=57, LONGRUN scope N=207, 23 excluded BWCN IDs, one unavailable LONGRUN ID, and available N=206.


In [ ]:
waccm_output = bootstrap_and_package(waccm_state, repetitions=5000, seed=15600)
master_threshold = float(waccm_rankings.low25_threshold_du.iloc[0])
if not np.isclose(float(waccm_output.attrs["low25_threshold_du"]), master_threshold):
    raise RuntimeError("Figure 15 WACCM did not inherit the 230-event master threshold")
excluded_ids = list(filter(None, str(waccm_output.attrs["excluded_master_event_ids"]).split(",")))
unavailable_ids = list(filter(None, str(waccm_output.attrs["unavailable_event_ids"]).split(",")))
if (
    int(waccm_output.attrs["master_sample_size"]) != 230
    or int(waccm_output.attrs["master_low_count"]) != 57
    or str(waccm_output.attrs["field_scope_segments"]) != "LONGRUN"
    or int(waccm_output.attrs["scope_master_count"]) != 207
    or int(waccm_output.attrs["available_event_count"]) != 206
    or int(waccm_output.attrs["available_low_count"]) != 51
    or int(waccm_output.attrs["available_event_count"]) != waccm_output.sizes["event_year"]
    or int(waccm_output.attrs["available_low_count"]) != int(waccm_output.is_low25.sum())
    or len(excluded_ids) != 23
    or not all(event_id.startswith("BWCN:") for event_id in excluded_ids)
    or unavailable_ids != ["LONGRUN:0001"]
):
    raise RuntimeError("Figure 15 WACCM master/availability bookkeeping is inconsistent")
write_figure15(waccm_output, "waccm_bootstrap5000.nc")
